## Test Grid SANOS



In [ ]:
%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.widb.sampler
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.torch
%aimport quantdata.torch.bbtorch
%aimport quantdata.torch.ffw
%aimport quantdata.dysanos.simple
%aimport quantdata.dysanos.simple.sampler
%aimport quantdata.dysanos.simple.model
%aimport quantdata.sanos.utils
%aimport quantdata.sanos.grid
%aimport quantdata.sanos.fitter

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"]).data['SPX']
print(f"/done; columns {data.keys()}")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ImportError: cannot import name 'Eps' from 'quantdata.sanos.utils' (/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/utils.py)

In [ ]:
%aimport quantdata.sanos.grid
%aimport quantdata.sanos.grid.fitter

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store
from cdxcore.verbose import Context
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.sanos.grid import GridGeometry, GridSanos, BackboneNumpy
from quantdata.sanos.fitter import Eps, Fitter
import numpy as np

def fit(data):
    num_samples = len(data.dates)

    # construct grid
    # --------------

    grid        = GridGeometry.create_grid_nms(
                            horizon_t  = 1., 
                            num_times  = 10,
                            nms_range  = (-4,4),
                            num_nms    = 60,
                            dlv_min    = 0.01,
                            dlv_max    = 5.,
                            fwdvol_min = 0.0,
                            vol_interp = "lin_var",
                            lower_scale = 0.2,
                            upper_scale = 1.5,
                            dtype = np.float64
            )

    eps         = Eps()
    fitter      = Fitter( geometry = grid, eps = eps, debug=True, max_error = None, mid_fit_weight = 0.1 )
    sanos       = GridSanos( geometry = grid, bb = BackboneNumpy() )

    def fit_sample( isample, store ):

        d          = data.dates[isample]
        is_valid   = data.is_valid[isample,:]
        strikes    = data.pure_strike[isample,is_valid]
        tte_y      = data.tte_y[isample,is_valid]
        bid_vol    = data.bid_vol[isample,is_valid]
        ask_vol    = data.ask_vol[isample,is_valid]
        bid        = data.bid_pure_price[isample,is_valid]
        ask        = data.ask_pure_price[isample,is_valid]

        # plot the grid
        # -------------

        with figure(f"Grid Geometry {d}", col_size=5, row_size=4, columns=None ) as fig_strikes:
            axD, axA    = fig_strikes.add_subplots("Inner", "All")
            axD.set_xlabel("TTE")
            axA.set_xlabel("TTE")
            axD.set_ylabel("Pure Strike (log-scale)")
            axA.set_ylabel("Pure Strike (log-scale)")
            axA.set_yscale("log")
            axD.set_yscale("log")

            for i in range(grid.N+2):
                axA.plot( grid.xexpiries, grid.xstrikes[:,i], ".:")
                if i>0 and i<grid.N-1:  
                    axD.plot( grid.xexpiries, grid.xstrikes[:,i], ".:" )
                    
            fig_strikes.render()
            store = fig_strikes.store()

            def plot_strikes(store):
                utty, inv = np.unique(tte_y, return_inverse=True)

                min_strike = np.full(len(utty), np.inf, dtype=strikes.dtype)
                max_strike = np.full(len(utty), -np.inf, dtype=strikes.dtype)

                np.minimum.at(min_strike, inv, strikes)
                np.maximum.at(max_strike, inv, strikes)
            
                store.clear()
                store += axA.fill_between(utty, min_strike, max_strike, alpha=0.3, color="blue")
                store += axD.fill_between(utty, min_strike, max_strike, alpha=0.3, color="blue")            
                store += axD.plot(utty, min_strike, ".-", color="blue")
                store += axD.plot(utty, max_strike, ".-", color="blue")
                store += axD.plot(tte_y, strikes, ".", color="blue")
                fig_strikes.render()

                return utty, min_strike, max_strike

            plot_strikes(store)


        # compute prices, dlvs
        # --------------------

        FWD_VOLS = 0.05

        dlvsqs, fit_prices, densities = \
            fitter( fwd_vols     = FWD_VOLS,
                    opt_strikes  = strikes,
                    opt_bid      = bid,
                    opt_ask      = ask,
                    opt_tte      = tte_y,
                    opt_is_calls = None,
                    opt_mask     = None,
                    return_only_dlvsqs = False,
                    verbose      = Context.all ) 
        assert fit_prices.shape == strikes.shape, f"Expected fit_prices to have shape {strikes.shape}, got {fit_prices.shape}"
        fit_vols  = bs.implied(strikes, fit_prices, sqrtT = np.sqrt(tte_y) )

        densities = np.asarray( densities, dtype=np.float64 )
        sanos = GridSanos( geometry=grid, bb=BackboneNumpy(), debug=True )

        re_prices = sanos.price(  fwd_vols=FWD_VOLS, densities=densities,
                                  opt_strikes  = strikes,
                                  opt_tte      = tte_y
                               )
        re_vols  = bs.implied(strikes, re_prices, sqrtT = np.sqrt(tte_y) )

        test_dens_from_dlv  = sanos.densities( dlvsqs )
        dlv_prices = sanos.price( fwd_vols=FWD_VOLS, densities=test_dens_from_dlv,
                                  opt_strikes  = strikes,
                                  opt_tte      = tte_y
                               )
        dlv_vols  = bs.implied(strikes, dlv_prices, sqrtT = np.sqrt(tte_y) )

        # DLVs, manual
        # ------------

        C_t   = np.maximum( grid.strikes[:,None,:] - grid.strikes[:,:,None], 0. ).astype(np.float64) * densities[:,None,:].astype(np.float64)
        C_t   = np.sum(C_t, axis=-1)
        C_tm1 = np.maximum( grid.strikes[:-1,None,:]- grid.strikes[1:,:,None], 0. ).astype(np.float64) * densities[1:,None,:].astype(np.float64)
        C_tm1 = np.sum(C_tm1, axis=-1)
        C_0m1 = np.maximum( grid.strikes[0,None,:] - grid.strikes[1,:,None], 0. ).astype(np.float64) * grid.density0[None,:].astype(np.float64)
        C_0m1 = np.sum(C_0m1, axis=-1)

        dC_t  = C_t[1:,:] - C_tm1
        dC_0  = C_t[0,:] - C_0m1

        # DLV formula us
        #
        # C(t,k^i) - \bar C(t,k^i) = 1/2 DLV^2 dt k^i^2 \partial^2 C / \partial k^2
        #
        # \partial^2 C / \partial k^2 = { dC_t - dC_tm1 }/{ 1/2 (K_{i+1} - K_{i-1}) } = p_t / { 1/2 (K_{i+1} - K_{i-1}) } 
        #
        # hence
        #
        # C(t,k^i) - \bar C(t-1,k^i) = DLV^2 dt {  k^i^2 / (K_{i+1} - K_{i-1}) } p_t

        dts      = np.diff( grid.xexpiries.astype(np.float64) )
        box      = grid.strikes[:,1:-1].astype(np.float64)**2 / ( grid.strikes[:,2:] - grid.strikes[:,:-2] ).astype(np.float64) * dts[:,None]
        dlvsqs2  = np.zeros_like(grid.strikes[:,1:-1])
        dlvsqs2[1:,:] = dC_t[:,1:-1] / ( box[1:,:] * densities[1:,1:-1] )
        dlvsqs2[0,:]  = dC_0[1:-1] / ( box[0,:] * densities[0,1:-1] )
        dlvsqs2 = np.where( np.isfinite(dlvsqs2), np.maximum( dlvsqs2 - (grid.dlv_min**2.), 0. ), 0. )

        sanos_dens2 = sanos.densities( dlvsqs2 )
        prices2  = sanos.price(  fwd_vols=FWD_VOLS, densities=sanos_dens2,
                                  opt_strikes  = strikes,
                                  opt_tte      = tte_y
                               )
        vols2  = bs.implied(strikes, prices2, sqrtT = np.sqrt(tte_y) )

        with figure(f"Fits {d}", col_size=5, row_size=4, columns=4 ) as fig_fits:

            uttys = np.unique(tte_y)

            for utty in uttys:
                mask = tte_y == utty
                ax = fig_fits.add_subplot(f"TTE {utty*DAYS_PER_YEAR:.3f}d")
                ax.set_xlabel("Pure Strike")
                ax.set_ylabel("Implied")

                istrike = strikes[mask]
                ixs     = np.argsort(istrike)

                ax.plot(istrike[ixs], bid_vol[mask][ixs], "^-", color="green", label="Bid")
                ax.plot(istrike[ixs], ask_vol[mask][ixs], "v-", color="red", label="Ask")
                ax.plot(istrike[ixs], re_vols[mask][ixs], "o-", color="purple", label="Repriced")
                ax.plot(istrike[ixs], dlv_vols[mask][ixs], "s-", color="blue", label="DLV")
                ax.plot(istrike[ixs], fit_vols[mask][ixs], ".:", color="orange", label="Fit")
                #ax.plot(istrike[ixs], vols2[mask][ixs], "*:", color="cyan", label="Manual DLV")
                ax.legend()

        if False:
            # reverse the calculation https://arxiv.org/pdf/2601.11209 page 19
            #
            # w^i=_t = 1/2 DLVSQ k^i^2 DT g^=_t
            #
            # g^+_t = 1/{ 1/2 (k^{i+1}-k^{i-1})(K^i+1 - K^i) }
            # g^-_t = 1/{ 1/2 (k^{i+1}-k^{i-1})(K^i - K^i-1) }
            #
        
            g_plus  = 1. / ( 0.5 * ( grid.strikes[:,2:] - grid.strikes[:,:-2] ) * ( grid.strikes[:,2:] - grid.strikes[:,1:-1] ) )
            g_minus = 1. / ( 0.5 * ( grid.strikes[:,2:] - grid.strikes[:,:-2] ) * ( grid.strikes[:,1:-1] - grid.strikes[:,:-2] ) )
            w_plus  = 0.5 * dlvsqs2 * grid.strikes[:,1:-1]**2 * dts[:,None] * g_plus
            w_minus = 0.5 * dlvsqs2 * grid.strikes[:,1:-1]**2 * dts[:,None] * g_minus
            xstrikes = grid.xstrikes
            tarK     = xstrikes[:-1, None, 1:-1]
            L        = ( np.maximum( tarK - xstrikes[1:,2:, None], 0. ) - np.maximum( tarK - xstrikes[1:,1:-1, None], 0. ) ) / ( xstrikes[1:,2:, None] - xstrikes[1:,1:-1, None] )\
                    -\
                    ( np.maximum( tarK - xstrikes[1:,1:-1, None], 0. ) - np.maximum( tarK - xstrikes[1:,:-2, None], 0. ) ) / ( xstrikes[1:,1:-1, None] - xstrikes[1:,:-2, None] )
            T, N = grid.TN
            assert L.shape == (T,N,N), f"Expected L to have shape {(T,N,N)}, got {L.shape}"

            from quantdata.sanos.tridiag import solve_sanos_tridiagonal

            density = grid.density0
            densities_manual = np.zeros((T,N))
            for t in range(T):
                density = L[t,:,:] @ density
                assert density.shape == (N,), f"Expected density to have shape {(N,)}, got {density.shape}"
                assert np.min(density)>-1E-8, f"Expected density to be non-negative, found min {np.min(density)}"
                assert abs(np.sum(density) - 1.) < 1E-4, f"Expected density to sum to 1, found {np.sum(density)-1.}"
                assert abs(np.sum(xstrikes[t+1,1:-1]@density) - 1.) < 1E-4, f"Expected density to exepct 1, found {np.sum(density)-1.}"


                density = solve_sanos_tridiagonal( backbone=BackboneNumpy(), w_minus=w_minus[t,:], w_plus=w_plus[t,:], rhs=density )
                assert density.shape == (N,), f"Expected density to have shape {(N,)}, got {density.shape}"
                assert np.min(density)>-1E-8, f"Expected density to be non-negative, found min {np.min(density)}"
                assert abs(np.sum(density) - 1.) < 1E-4, f"Expected density to sum to 1, found {np.sum(density)-1.}"
                assert abs(np.sum(xstrikes[t+1,1:-1 ]@density) - 1.) < 1E-4, f"Expected density to exepct 1, found {np.sum(density)-1.}"

                densities_manual[t,:] = density

            # Double check DLV pricing
            # ------------------------

            with figure("Fits", col_size=5, row_size=4, columns=4 ) as fig_fits:

                for t in range(grid.T):
                    strikes_t   = grid.strikes[t,:]

                    ax = fig_fits.add_subplot(f"TTE {grid.expiries[t]*DAYS_PER_YEAR:.3f}d")
                    ax.set_xlabel("Pure Strike")
                    ax.set_ylabel("Density")
                    ax.plot( strikes_t, densities[t,:], "*", color="blue", label="Sanos" )
                    ax.plot( strikes_t, test_dens_from_dlv[t,:], ".-", color="orange", label="From Sanos DLV" )
                    ax.plot( strikes_t, sanos_dens2[t,:], ".-", color="red", label="From manual DLV" )
                    ax.plot( strikes_t, densities_manual[t,:], ".-", color="purple", label="All manual" )
                    ax.legend(loc="upper right")


    for isample in range(num_samples-1,num_samples-200,-20):
        fit_sample(isample, store)

fit(data)


ImportError: cannot import name 'Eps' from 'quantdata.sanos.utils' (/mnt/c/Users/hans/OneDrive/Python3/quant_data/quantdata/sanos/utils.py)